# Learning 5: Your First Graph

**Goal**: Build your first LangGraph workflow

## What You'll Learn
- Core LangGraph concepts: State, Nodes, Edges
- Building a simple graph
- Compiling and running graphs
- Visualizing graph structure

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langgraph.graph import StateGraph, START, END
from typing import TypedDict

print("Setup complete!")

## LangGraph Core Concepts

Think of LangGraph as a flowchart:

1. **State**: Data that flows through the graph (like water in pipes)
2. **Nodes**: Processing steps (like stations that modify the data)
3. **Edges**: Connections between nodes (like pipes)

```
[START] → [Node A] → [Node B] → [END]
            ↓            ↑
         state        state
```

## Step 1: Define the State

State is a dictionary that holds all data flowing through your graph.

In [ ]:
# Define state using TypedDict
class SimpleState(TypedDict):
    name: str
    greeting: str

# Example of what state looks like
example_state: SimpleState = {
    "name": "Alice",
    "greeting": ""
}
print("Example state:", example_state)

## Step 2: Create Node Functions

Nodes are functions that:
- Take the current state as input
- Return updates to the state

In [ ]:
def greet(state: SimpleState) -> dict:
    """Create a greeting for the user."""
    name = state["name"]
    return {"greeting": f"Hello, {name}! Welcome!"}

# Test the function
test_state = {"name": "Bob", "greeting": ""}
result = greet(test_state)
print("Node output:", result)

## Step 3: Build the Graph

Connect nodes with edges to create the workflow.

In [ ]:
# Create the graph
graph_builder = StateGraph(SimpleState)

# Add nodes
graph_builder.add_node("greeter", greet)

# Add edges
graph_builder.add_edge(START, "greeter")  # START → greeter
graph_builder.add_edge("greeter", END)     # greeter → END

print("Graph structure defined!")

## Step 4: Compile the Graph

Compiling creates an executable application from your graph definition.

In [ ]:
# Compile
app = graph_builder.compile()

print("Graph compiled successfully!")
print("Type:", type(app))

## Step 5: Run the Graph

In [ ]:
# Run with input state
result = app.invoke({"name": "Alice", "greeting": ""})

print("Final state:", result)
print("Greeting:", result["greeting"])

## Visualize the Graph

LangGraph can generate a visual representation of your graph.

In [ ]:
from IPython.display import Image, display

# Display the graph (requires graphviz)
try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Graph visualization:")
    print(app.get_graph().draw_ascii())

## A Multi-Node Graph

Let's build a more interesting graph with multiple steps.

In [ ]:
class ProcessingState(TypedDict):
    text: str
    word_count: int
    uppercase: str
    summary: str

def count_words(state: ProcessingState) -> dict:
    """Count words in the text."""
    count = len(state["text"].split())
    return {"word_count": count}

def make_uppercase(state: ProcessingState) -> dict:
    """Convert text to uppercase."""
    return {"uppercase": state["text"].upper()}

def create_summary(state: ProcessingState) -> dict:
    """Create a summary of the processing."""
    summary = f"Text has {state['word_count']} words. Uppercase version: {state['uppercase']}"
    return {"summary": summary}

In [ ]:
# Build the multi-node graph
builder = StateGraph(ProcessingState)

# Add all nodes
builder.add_node("counter", count_words)
builder.add_node("uppercaser", make_uppercase)
builder.add_node("summarizer", create_summary)

# Connect them: START → counter → uppercaser → summarizer → END
builder.add_edge(START, "counter")
builder.add_edge("counter", "uppercaser")
builder.add_edge("uppercaser", "summarizer")
builder.add_edge("summarizer", END)

# Compile and run
app = builder.compile()

result = app.invoke({
    "text": "Hello world this is a test",
    "word_count": 0,
    "uppercase": "",
    "summary": ""
})

print("Final result:")
for key, value in result.items():
    print(f"  {key}: {value}")

In [ ]:
# Visualize
try:
    display(Image(app.get_graph().draw_mermaid_png()))
except:
    print(app.get_graph().draw_ascii())

## Streaming Graph Execution

Watch the graph execute step by step.

In [ ]:
# Stream to see each step
for step in app.stream({
    "text": "LangGraph is awesome",
    "word_count": 0,
    "uppercase": "",
    "summary": ""
}):
    print("Step:", step)
    print("---")

## Exercise: Build Your Own Graph

Create a graph that:
1. Takes a number as input
2. Doubles it in one node
3. Adds 10 in another node
4. Formats it as a string in a final node

In [ ]:
# Your code here!



## Key Takeaways

1. **State** = TypedDict that holds all graph data
2. **Nodes** = Functions that process and update state
3. **Edges** = Connections between nodes
4. `START` and `END` are special built-in nodes
5. `.compile()` creates an executable app
6. `.invoke()` runs the graph, `.stream()` shows each step

**Next**: Learning 6 - Conditional Edges